# Notebook 08 — Validación cruzada multilingüe de series temporales NDVI

Compara dos extracciones independientes de la serie temporal NDVI sobre las 8 estaciones de
muestreo INVEMAR:

1. **Python + GEE** — flujo de la Fase 2: `reduceRegions` sobre la colección Sentinel-2
   en la nube, exportado como `serie_temporal_ndvi_definitiva.csv`.
2. **R + stars** — flujo del Cap. 21: cubo construido localmente desde los TIFs
   trimestrales, extraído con `st_extract`, exportado como `series_temporales_stars.csv`.

Si las dos series coinciden dentro de un margen razonable (RMSE < 0.05 unidades NDVI), se
confirma que el resultado no depende de la implementación. Discrepancias mayores indican
sesgos metodológicos (diferencias de máscara de nubes, de remuestreo, o del manejo de
píxeles mixtos en la frontera del polígono de extracción).


In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT   = Path('..').resolve()
TABLES = ROOT / 'outputs' / 'tables'
FIGS   = ROOT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

CSV_PYTHON = TABLES / 'serie_temporal_ndvi_definitiva.csv'
CSV_R      = TABLES / 'series_temporales_stars.csv'

for p in (CSV_PYTHON, CSV_R):
    if not p.exists():
        print(f'Falta: {p}')
    else:
        print(f'OK: {p}')


## 1. Cargar y normalizar las dos series


In [ ]:
# Python + GEE: cols [subzona, date, ndvi]
df_py = pd.read_csv(CSV_PYTHON)
df_py = df_py.rename(columns={'subzona':'estacion', 'date':'fecha'})
df_py['fecha'] = pd.to_datetime(df_py['fecha'])
df_py['fuente_extraccion'] = 'Python+GEE'

# R + stars: cols [estacion, fuente, fecha, ndvi]
df_r = pd.read_csv(CSV_R)
df_r['fecha'] = pd.to_datetime(df_r['fecha'])
df_r['fuente_extraccion'] = 'R+stars'

print(f'Python+GEE: {len(df_py)} filas, {df_py["estacion"].nunique()} estaciones')
print(f'R+stars:    {len(df_r)} filas, {df_r["estacion"].nunique()} estaciones')


## 2. Pareo por (estación, mes) y diagnóstico


In [ ]:
# Para que sean comparables, agregar a granularidad mensual
def mes(df):
    df = df.copy()
    df['anio_mes'] = df['fecha'].dt.to_period('M').dt.to_timestamp()
    return df.groupby(['estacion','anio_mes'])['ndvi'].mean().reset_index()

py_mes = mes(df_py).rename(columns={'ndvi':'ndvi_python'})
r_mes  = mes(df_r ).rename(columns={'ndvi':'ndvi_r'})

cmp = py_mes.merge(r_mes, on=['estacion','anio_mes'], how='inner').dropna()
cmp['diff']    = cmp['ndvi_python'] - cmp['ndvi_r']
cmp['absdiff'] = cmp['diff'].abs()
print(f'Pares comparables: {len(cmp)}')
print(cmp.describe()[['ndvi_python','ndvi_r','diff','absdiff']])


In [ ]:
# Métricas globales
rmse  = float(np.sqrt((cmp['diff']**2).mean()))
mae   = float(cmp['absdiff'].mean())
bias  = float(cmp['diff'].mean())
rho   = float(cmp['ndvi_python'].corr(cmp['ndvi_r']))
print(f'RMSE = {rmse:.4f}')
print(f'MAE  = {mae:.4f}')
print(f'Bias = {bias:+.4f}  (positivo: Python > R)')
print(f'ρ    = {rho:.4f}')

# Tabla por estación
por_est = (cmp.groupby('estacion')
              .agg(n=('diff','count'),
                   bias=('diff','mean'),
                   rmse=('diff', lambda s: float(np.sqrt((s**2).mean()))),
                   rho=('diff', lambda s: cmp.loc[s.index].pipe(
                        lambda d: d['ndvi_python'].corr(d['ndvi_r']))))
              .round(4))
por_est.to_csv(TABLES / 'validacion_multilingual.csv')
print('\nGuardado: outputs/tables/validacion_multilingual.csv')
por_est


## 3. Gráfico de dispersión Python vs R


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(cmp['ndvi_r'], cmp['ndvi_python'], s=14, alpha=0.5, color='#2c7fb8')
lims = (max(0, cmp[['ndvi_python','ndvi_r']].min().min()-0.05),
        min(1, cmp[['ndvi_python','ndvi_r']].max().max()+0.05))
ax.plot(lims, lims, 'k--', linewidth=0.8, label='1:1')
ax.set_xlabel('NDVI (R + stars, Cap. 21)')
ax.set_ylabel('NDVI (Python + GEE, Fase 2)')
ax.set_title(f'Validación cruzada multilingüe — ρ = {rho:.3f}, RMSE = {rmse:.3f}')
ax.legend()
ax.set_xlim(lims); ax.set_ylim(lims)
fig.tight_layout()
out = FIGS / 'validacion_multilingual_scatter.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Guardado: {out}')


## 4. Lectura

Si **ρ > 0.95 y RMSE < 0.05**, las dos implementaciones son funcionalmente equivalentes
y el flujo Python+GEE de la Fase 2 queda validado contra una implementación local
e independiente con stars. Esta es la validación cruzada multilingüe que justifica el
uso del pipeline tri-lenguaje del proyecto: cada lenguaje opera en su zona de comodidad
(Python en la nube con GEE, R sobre cubo local con stars, Julia sobre vectores con
GeoJSON.jl) pero los resultados son comparables.

Si la **discrepancia es mayor**, las causas más probables son: (i) máscara de nubes
diferente entre el QA60 aplicado en GEE y los TIFs ya enmascarados; (ii) diferencias
de remuestreo al ensamblar el composite trimestral; (iii) píxeles mixtos en la frontera
del buffer alrededor de cada estación.
